# ADF and KPSS Stationarity Tests

**Week 2 — Stationarity | Notebook 2 of 3**

This notebook applies formal stationarity tests to **5 different series** and demonstrates the
**ADF + KPSS four-quadrant decision matrix**:

| | KPSS fails to reject (stationary) | KPSS rejects (non-stationary) |
|---|---|---|
| **ADF rejects (stationary)** | STATIONARY | TREND-STATIONARY |
| **ADF fails to reject (non-stat.)** | INCONCLUSIVE | NON-STATIONARY |

The key insight: ADF and KPSS have **opposite null hypotheses**, so using both together gives
a more robust verdict than either alone.

In [ ]:
import sys
sys.path.insert(0, "../..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from week__01__fundamentals.ts_loader import TimeSeriesLoader
from week__02__stationarity.stationarity_tests import StationarityTester, StationarityVerdict

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 3)

tester = StationarityTester(alpha=0.05)

## 1. Prepare the 5 Test Series

| # | Series | Expected Verdict | Why |
|---|--------|-----------------|-----|
| 1 | Airline Passengers | NON_STATIONARY | Trend + growing variance |
| 2 | Sunspots | STATIONARY | No persistent trend |
| 3 | CO2 (Mauna Loa) | NON_STATIONARY | Strong trend |
| 4 | Synthetic Random Walk | NON_STATIONARY | Unit root by construction |
| 5 | Synthetic White Noise | STATIONARY | i.i.d. by construction |

In [ ]:
# Real-world series
airline, _ = TimeSeriesLoader("AirPassengers").from_statsmodels("airline")
sunspots, _ = TimeSeriesLoader("Sunspots").from_statsmodels("sunspots")
co2, _ = TimeSeriesLoader("CO2").from_statsmodels("co2")

# Synthetic series (fixed seed for reproducibility)
rng = np.random.default_rng(seed=42)
n = 500
dates = pd.date_range("2000-01-01", periods=n, freq="D")

white_noise = pd.Series(rng.standard_normal(n), index=dates, name="WhiteNoise")
random_walk = pd.Series(
    np.cumsum(rng.standard_normal(n)), index=dates, name="RandomWalk"
)

all_series = {
    "Airline Passengers": airline,
    "Sunspots": sunspots,
    "CO2": co2,
    "Random Walk": random_walk,
    "White Noise": white_noise,
}

## 2. Visual Overview

In [ ]:
fig, axes = plt.subplots(len(all_series), 1, figsize=(14, 3 * len(all_series)))

for ax, (name, series) in zip(axes, all_series.items()):
    ax.plot(series, linewidth=0.8)
    ax.set_title(name)

plt.tight_layout()
plt.show()

## 3. Run ADF + KPSS on All 5 Series

In [ ]:
results = {}

for name, series in all_series.items():
    result = tester.fit(series)
    results[name] = result
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")
    print(result)

## 4. Summary Table

In [ ]:
summary_rows = []
for name, r in results.items():
    summary_rows.append({
        "Series": name,
        "ADF p-value": f"{r.adf.p_value:.4f}",
        "ADF rejects?": "Yes" if r.adf.reject_null else "No",
        "KPSS p-value": f"{r.kpss.p_value:.4f}",
        "KPSS rejects?": "Yes" if r.kpss.reject_null else "No",
        "Verdict": r.verdict.name,
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

## 5. Interpreting the Results

### Airline Passengers → NON_STATIONARY
- ADF fails to reject the unit root (p ≈ 0.99) — evidence of non-stationarity
- KPSS rejects stationarity (p < 0.05) — confirms non-stationarity
- Both tests agree: this series needs differencing (and a log transform for the variance)

### Sunspots → STATIONARY
- ADF rejects the unit root — no evidence of a unit root
- KPSS fails to reject stationarity — consistent with stationarity
- Expected: sunspots are a classic stationary cyclical series

### CO2 → NON_STATIONARY
- Strong upward trend. ADF cannot reject the unit root.
- KPSS rejects stationarity.
- This series needs at least first differencing, possibly seasonal differencing too.

### Random Walk → NON_STATIONARY
- A random walk is a unit root process by construction (yₜ = yₜ₋₁ + εₜ)
- ADF should fail to reject (it does) and KPSS should reject (it does)
- First differencing (d=1) will turn this into white noise

### White Noise → STATIONARY
- i.i.d. draws have no autocorrelation, constant mean and variance
- ADF rejects the unit root, KPSS fails to reject stationarity
- No transformation needed — this is the ideal residual for a fitted model

## 6. Demonstrating ADF + KPSS Disagreement

To show the **TREND_STATIONARY** case (ADF rejects but KPSS also rejects), we construct
a series with a deterministic trend but no unit root: `yₜ = 0.5t + εₜ`.

This is trend-stationary: the correct remedy is **detrending** (subtracting the trend),
not differencing.

In [ ]:
# Construct a trend-stationary series
trend_stationary = pd.Series(
    0.5 * np.arange(n) + rng.standard_normal(n) * 10,
    index=dates,
    name="TrendStationary",
)

fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(trend_stationary, linewidth=0.8)
ax.set_title("Trend-Stationary Series: yₜ = 0.5t + εₜ")
plt.show()

ts_result = tester.fit(trend_stationary)
print(ts_result)

**Interpretation:**
- ADF may reject (no unit root — correct, this series has a deterministic trend, not a stochastic one)
- KPSS rejects stationarity (the trend causes the mean to drift)
- Verdict: TREND_STATIONARY — the correct action is to detrend, not to difference.
  Differencing a trend-stationary series introduces unnecessary MA structure.

## 7. Phillips-Perron (PP) Test as Cross-Check

The PP test has the **same null hypothesis as ADF** (unit root exists) but handles serial correlation
and heteroscedasticity differently — it uses a non-parametric correction instead of adding lagged
difference terms. This makes it useful as a cross-check when:
- ADF's lag selection (via AIC) might be unreliable
- The series has conditional heteroscedasticity (ARCH effects)

**Note:** PP requires the `arch` package. If not installed, this section can be skipped.

In [ ]:
try:
    from arch.unitroot import PhillipsPerron

    pp_tester = StationarityTester(alpha=0.05)

    pp_rows = []
    for name, series in all_series.items():
        result = pp_tester.fit(series, run_pp=True)
        pp_rows.append({
            "Series": name,
            "ADF p-value": f"{result.adf.p_value:.4f}",
            "PP p-value": f"{result.pp.p_value:.4f}",
            "ADF agrees with PP?": "Yes" if result.adf.reject_null == result.pp.reject_null else "No",
            "PP rejects unit root?": "Yes" if result.pp.reject_null else "No",
        })

    pp_df = pd.DataFrame(pp_rows)
    print("ADF vs Phillips-Perron comparison:")
    print(pp_df.to_string(index=False))

except ImportError:
    print("arch package not installed — skipping PP test demo.")
    print("Install with: pip install arch")

**PP interpretation:**
- For all 5 series, PP should agree with ADF — they test the same null hypothesis with different methods.
- When they agree, confidence in the verdict increases.
- When they disagree (rare), it usually means the series has unusual error structure (e.g., strong ARCH effects) — investigate further before differencing.

## 8. Structural Break Awareness

A **structural break** is a sudden, permanent shift in the mean or trend of a series — e.g., a policy
change, a market crash, or a regime switch. Structural breaks can fool stationarity tests:

- **ADF may fail to reject** (interprets the break as evidence of a unit root)
- **KPSS will reject** (the mean clearly shifts)
- Result: looks like NON_STATIONARY, but **differencing is the wrong fix** — it spreads the break across
  the entire series instead of isolating it.

The correct approach: detect the break, model the two regimes separately (or include a dummy variable).

Below we construct a synthetic example: white noise with a level shift at the midpoint.

In [ ]:
# Construct: white noise with a structural break (level shift at midpoint)
half = n // 2
break_series = pd.Series(
    np.concatenate([
        rng.standard_normal(half),                    # regime 1: mean ≈ 0
        rng.standard_normal(n - half) + 5.0,          # regime 2: mean ≈ 5
    ]),
    index=dates,
    name="StructuralBreak",
)

fig, axes = plt.subplots(1, 2, figsize=(16, 4))

# Plot the series with the break highlighted
axes[0].plot(break_series, linewidth=0.6)
axes[0].axvline(x=dates[half], color="red", linestyle="--", linewidth=2, label="Break point")
axes[0].set_title("White Noise + Structural Break (level shift of +5 at midpoint)")
axes[0].legend()

# Show what differencing does to a break — creates a spike artifact
diffed_break = break_series.diff().dropna()
axes[1].plot(diffed_break, linewidth=0.6)
axes[1].axvline(x=dates[half], color="red", linestyle="--", linewidth=2, label="Break point")
axes[1].set_title("After differencing — the break becomes a spike, not a fix")
axes[1].legend()

plt.tight_layout()
plt.show()

# Run ADF + KPSS on the break series
break_result = tester.fit(break_series)
print("Full series (with break):")
print(break_result)

# Now test each regime separately — both should be stationary
print(f"\nRegime 1 (before break):")
print(tester.fit(break_series.iloc[:half]))
print(f"\nRegime 2 (after break):")
print(tester.fit(break_series.iloc[half:]))

## 9. Key Takeaways

1. **Always run both ADF and KPSS** — their opposite null hypotheses complement each other.
2. When they **agree** (both say stationary or both say non-stationary), the verdict is clear.
3. When they **disagree**, investigate: trend-stationarity and structural breaks are the usual causes.
4. **PP test** is a useful cross-check alongside ADF — same null hypothesis, different error handling.
5. **Structural breaks** can fool all tests — always inspect the series visually before differencing.
6. The four-quadrant matrix is a decision framework, not a black box — always inspect the series visually.

**Next:** Notebook 3 applies transformations and differencing to make non-stationary series stationary.

## 7. Key Takeaways

1. **Always run both ADF and KPSS** — their opposite null hypotheses complement each other.
2. When they **agree** (both say stationary or both say non-stationary), the verdict is clear.
3. When they **disagree**, investigate: trend-stationarity and structural breaks are the usual causes.
4. The four-quadrant matrix is a decision framework, not a black box — always inspect the series visually.

**Next:** Notebook 3 applies transformations and differencing to make non-stationary series stationary.